In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
load_dotenv()

import os
import sys
sys.path.append(os.path.abspath("../../"))


ROOT_DIR = os.getenv("ROOT_DIR")
INPUT_TRAIN = os.path.join(ROOT_DIR, ".data_xuetangx/XuetangX/processed/xuetang_flat_train.csv")
OUTPUT_DIR = os.path.join(ROOT_DIR, "outputs/eda_xuetang")
PLOT_DIR = os.path.join(ROOT_DIR, "plots/eda_xuetang")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

In [18]:
df_flat = pd.read_csv(INPUT_TRAIN)

metadata_cols = ['global_id', 'day', 'dropout']
action_cols = [col for col in df_flat.columns if col not in metadata_cols]

agg_funcs = {col: 'sum' for col in action_cols}
agg_funcs['dropout'] = 'first'

df_agg = df_flat.groupby('global_id').agg(agg_funcs).reset_index()

In [19]:
# Analisi dello sbilanciamento
class_counts = df_agg['dropout'].value_counts()
class_pct = df_agg['dropout'].value_counts(normalize=True) * 100

print("[INFO] Distribuzione Dropout (1) vs Persist (0):")
for idx, count in class_counts.items():
    print(f"Classe {idx}: {count} studenti ({class_pct[idx]:.1f}%)")

plt.figure(figsize=(8, 6))
sns.countplot(data=df_agg, x='dropout', palette=['forestgreen', 'indianred'])
plt.title("Distribuzione delle Classi (Train Set)", fontsize=14)
plt.xlabel("0 = Persist, 1 = Dropout", fontsize=12)
plt.ylabel("Numero di Studenti", fontsize=12)

plot_path = os.path.join(PLOT_DIR, "class_distribution.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Plot salvato in {plot_path}")

[INFO] Distribuzione Dropout (1) vs Persist (0):
Classe 1: 119817 studenti (75.9%)
Classe 0: 38126 studenti (24.1%)
[OK] Plot salvato in /Volumes/T7/documents/github/dropout-prediction/plots/eda_xuetang/class_distribution.png


/var/folders/qt/qm1533ls7gb9m_g8kg_dx1f00000gn/T/ipykernel_41138/1839497503.py:10: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df_agg, x='dropout', palette=['forestgreen', 'indianred'])


In [20]:
print("[INFO] Ricerca delle Dead Actions (azioni mai usate nel Train Set)...")

action_sums = df_agg[action_cols].sum()
dead_actions = action_sums[action_sums == 0].index.tolist()

print(f"[INFO] Trovate {len(dead_actions)} azioni morte su {len(action_cols)} totali.")

# Salvataggio CSV
pd.DataFrame({'dead_actions': dead_actions}).to_csv(os.path.join(OUTPUT_DIR, "dead_actions.csv"), index=False)

# Rimuoviamo le dead actions dal dataframe aggregato per le analisi successive
active_cols = [col for col in action_cols if col not in dead_actions]
df_agg_active = df_agg[['global_id', 'dropout'] + active_cols]

print(f"[SUCCESS] Feature attive rimaste: {len(active_cols)}")

[INFO] Ricerca delle Dead Actions (azioni mai usate nel Train Set)...
[INFO] Trovate 0 azioni morte su 22 totali.
[SUCCESS] Feature attive rimaste: 22


In [22]:
print("[INFO] Calcolo delle Top Actions...")

# Calcoliamo la percentuale di studenti che ha fatto un'azione ALMENO una volta
n_students = len(df_agg_active)
students_did_action = (df_agg_active[active_cols] > 0).sum()
pct_students = (students_did_action / n_students) * 100

top_actions = pct_students.sort_values(ascending=False)

# Salvataggio CSV
df_top = top_actions.reset_index()
df_top.columns = ['action', 'pct_students']
df_top.to_csv(os.path.join(OUTPUT_DIR, "top_actions.csv"), index=False)

# Plot delle prime 15
plt.figure(figsize=(12, 8))
sns.barplot(x=top_actions.head(15).values, y=top_actions.head(15).index, palette="viridis")
plt.title("Top 15 Azioni (% di studenti che le hanno eseguite almeno una volta)", fontsize=14)
plt.xlabel("% Studenti", fontsize=12)
plt.ylabel("Azione", fontsize=12)
plt.tight_layout()

plot_path = os.path.join(PLOT_DIR, "top_actions.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Plot salvato in {plot_path}")

[INFO] Calcolo delle Top Actions...
[OK] Plot salvato in /Volumes/T7/documents/github/dropout-prediction/plots/eda_xuetang/top_actions.png


/var/folders/qt/qm1533ls7gb9m_g8kg_dx1f00000gn/T/ipykernel_41138/1373164069.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_actions.head(15).values, y=top_actions.head(15).index, palette="viridis")


In [24]:
print("[INFO] Calcolo correlazione Point-Biserial (Azione -> Dropout)...")

# Calcolo correlazione solo con la colonna dropout
correlations = df_agg_active[active_cols + ['dropout']].corr()['dropout'].drop('dropout')
correlations = correlations.dropna().sort_values()

# Salvataggio CSV
corr_df = correlations.reset_index()
corr_df.columns = ['action', 'correlation']
corr_df.to_csv(os.path.join(OUTPUT_DIR, "action_dropout_correlation.csv"), index=False)

# Plot (Top 10 Negative e Top 10 Positive/Meno Negative)
plot_data = pd.concat([correlations.head(10), correlations.tail(10)])

plt.figure(figsize=(12, 8))
colors = ['forestgreen' if val < 0 else 'indianred' for val in plot_data.values]
sns.barplot(x=plot_data.values, y=plot_data.index, palette=colors)
plt.title("Correlazione Azione-Dropout (Top estremi)", fontsize=14)
plt.xlabel("Correlazione (Negativo = Riduce Dropout)", fontsize=12)
plt.axvline(0, color='black', linestyle='--')
plt.tight_layout()

plot_path = os.path.join(PLOT_DIR, "correlation_dropout.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Plot salvato in {plot_path}")

[INFO] Calcolo correlazione Point-Biserial (Azione -> Dropout)...
[OK] Plot salvato in /Volumes/T7/documents/github/dropout-prediction/plots/eda_xuetang/correlation_dropout.png


/var/folders/qt/qm1533ls7gb9m_g8kg_dx1f00000gn/T/ipykernel_41138/2664832679.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=plot_data.values, y=plot_data.index, palette=colors)


In [26]:
print("[INFO] Analisi dell'Engagement totale...")

# Somma di tutte le azioni per studente
df_agg_active['total_engagement'] = df_agg_active[active_cols].sum(axis=1)

eng_stats = df_agg_active.groupby('dropout')['total_engagement'].describe()
eng_stats.to_csv(os.path.join(OUTPUT_DIR, "engagement_stats.csv"))
print(eng_stats)

plt.figure(figsize=(8, 6))
sns.boxplot(data=df_agg_active, x='dropout', y='total_engagement', palette=['forestgreen', 'indianred'])
plt.title("Distribuzione Engagement Totale per Classe", fontsize=14)
plt.xlabel("0 = Persist, 1 = Dropout", fontsize=12)
plt.ylabel("Azioni Totali", fontsize=12)

# Taglio degli outlier visivi dinamico (95° percentile)
y_max = df_agg_active['total_engagement'].quantile(0.95)
plt.ylim(0, y_max)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plot_path = os.path.join(PLOT_DIR, "engagement_distribution.png")
plt.savefig(plot_path, dpi=300)
plt.close()
print(f"[OK] Plot salvato in {plot_path}")

[INFO] Analisi dell'Engagement totale...
            count        mean          std  min   25%    50%    75%       max
dropout                                                                      
0         38126.0  375.527383  1018.250730  0.0  27.0  163.0  447.0   58877.0
1        119817.0   83.630178   600.887885  0.0   4.0   15.0   53.0  128992.0
[OK] Plot salvato in /Volumes/T7/documents/github/dropout-prediction/plots/eda_xuetang/engagement_distribution.png


/var/folders/qt/qm1533ls7gb9m_g8kg_dx1f00000gn/T/ipykernel_41138/2216173912.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_agg_active['total_engagement'] = df_agg_active[active_cols].sum(axis=1)
/var/folders/qt/qm1533ls7gb9m_g8kg_dx1f00000gn/T/ipykernel_41138/2216173912.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_agg_active, x='dropout', y='total_engagement', palette=['forestgreen', 'indianred'])


In [28]:
print("[INFO] Calcolo Collinearità (feature troppo simili tra loro)...")

corr_matrix = df_agg_active[active_cols].corr()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

pairs = []
for col in upper_tri.columns:
    for row in upper_tri.index:
        val = upper_tri.loc[row, col]
        if pd.notna(val) and abs(val) > 0.75: # Soglia impostata a 0.75
            pairs.append({'feature_1': row, 'feature_2': col, 'correlation': round(val, 4)})

if pairs:
    df_pairs = pd.DataFrame(pairs).sort_values(by='correlation', ascending=False)
    df_pairs.to_csv(os.path.join(OUTPUT_DIR, "high_collinearity.csv"), index=False)
    print(f"[OK] Trovate {len(pairs)} coppie collineari (|corr| > 0.75).")
    
    cols_to_plot = list(set(df_pairs['feature_1']).union(set(df_pairs['feature_2'])))
    plt.figure(figsize=(12, 10))
    sns.heatmap(df_agg_active[cols_to_plot].corr(), annot=True, fmt=".2f", cmap="coolwarm", square=True)
    plt.title("Heatmap Collinearità (|corr| > 0.75)", fontsize=14)
    plt.tight_layout()
    
    plot_path = os.path.join(PLOT_DIR, "collinearity_heatmap.png")
    plt.savefig(plot_path, dpi=300)
    plt.close()
else:
    print("[INFO] Nessuna coppia altamente collineare trovata.")

[INFO] Calcolo Collinearità (feature troppo simili tra loro)...
[OK] Trovate 6 coppie collineari (|corr| > 0.75).
